# Optimization

### Resources
- https://github.com/automl/auto-sklearn/issues/1684

## Install auto-sklearn

In [ ]:
# 1. uninstall all affected packages
!pip uninstall -y Cython scipy pyparsing scikit_learn imbalanced-learn mlxtend yellowbrick

In [ ]:
# 2. install packages to be downgraded
!pip install Cython==0.29.36 scipy==1.9 pyparsing==2.4

In [ ]:
# 3. install older scikit-learn disregarding its dependencies
!pip install scikit-learn==0.24.2 --no-build-isolation

In [ ]:
# 4. finally install auto-sklearn
!pip install auto-sklearn

In [ ]:
# 5. then, try loading the package repeatedly until trash in its dependencies are clean
# import autosklearn

In [ ]:
!pip install pipelineprofiler

## Setup

In [ ]:
from pprint import pprint

import autosklearn.classification
import matplotlib.pyplot as plt
import pandas as pd
import PipelineProfiler
import seaborn as sns
from google.colab import drive
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/land_cover_classification_kaza

## Load train and test set

In [ ]:
train = pd.read_csv("data/train.csv")
train

In [ ]:
X_train = train.drop(["LC_Nr", "LC_Out", "Landcover"], axis=1)
X_train

In [ ]:
y_train = train["LC_Nr"]
y_train

In [ ]:
test = pd.read_csv("data/test.csv")
test

In [ ]:
X_test = test.drop(["LC_Nr", "LC_Out", "Landcover"], axis=1)
X_test

In [ ]:
y_test = test["LC_Nr"]
y_test

## Use auto-sklearn to find the best model

In [ ]:
automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=60 * 60,
    resampling_strategy="cv",
    resampling_strategy_arguments={"folds": 10},
    metric=autosklearn.metrics.f1_macro,
    n_jobs=-1,
    ensemble_kwargs={"ensemble_size": 0},
    include={"classifier": ["random_forest"], "feature_preprocessor": ["no_preprocessing"]},
)

In [ ]:
automl.fit(X_train, y_train)

In [ ]:
y_train_pred = automl.predict(X_train)
y_test_pred = automl.predict(X_test)

## Evaluation

### Train set classification metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_train, y_train_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_train, y_train_pred, average="macro")))

In [ ]:
print(classification_report(y_train, y_train_pred))

In [ ]:
cm = confusion_matrix(y_train, y_train_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

### Test set classifiction metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_test, y_test_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_test, y_test_pred, average="macro")))

In [ ]:
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

### Sprint statistics, model ranking and optimal hyperparameters

In [ ]:
print(automl.sprint_statistics())

In [ ]:
pprint(automl.show_models(), indent=4)

In [ ]:
automl.leaderboard(detailed=True, ensemble_only=False)

In [ ]:
automl.get_models_with_weights()

## Pipeline profiler

In [ ]:
profiler_data = PipelineProfiler.import_autosklearn(automl)
PipelineProfiler.plot_pipeline_matrix(profiler_data)

In [ ]:
# export has to be manually triggered in the gui before
PipelineProfiler.get_exported_pipelines()